# LEO Satellite Network Temporal Congestion Prediction & Embedding Generation
## LSTM Training Notebook using Raw Features + GAT Spatial Node Embeddings

This notebook implements the complete, leak-free training, validation, testing, evaluation, plotting, and **Temporal Embedding Export** pipeline for predicting satellite congestion scores ($\text{congestion\_score}(t+1)$).

### Pipeline Flow:
1. **Input Data**: Raw satellite features from `datasets/lstm_all_scenarios.csv` + 128-D GAT Spatial Embeddings from `artifacts/gat/spatial/embeddings/`.
2. **Feature Fusion**: Combines 24 raw features + 128 GAT spatial embeddings per timestep $\to$ **152-dimensional input feature sequences** $[30, 152]$ per satellite.
3. **LSTM Model**: 2-layer LSTM model that processes input sequences $[30, 152]$ and produces:
   - **Predicted Congestion**: Scalar prediction $\hat{y}(t+1)$.
   - **Temporal Node Embedding**: 128-dimensional temporal representation vector $h_{\text{temporal}} \in \mathbb{R}^{128}$ from the final LSTM hidden state.
4. **Output Export**: Exports all generated **128-D Temporal Node Embeddings** to `artifacts/lstm_gat_fusion/embeddings/` for downstream **PPO Reinforcement Learning Dynamic Routing**.

In [ ]:
import os
from pathlib import Path
import random
import time
import pickle
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

# Set global random seed for exact reproducibility
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True

set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch Version: {torch.__version__}")
print(f"Execution Device: {device}")

In [ ]:
# Directory & Path Configuration
BASE_DIR = Path("d:/Final year project 2") if Path("d:/Final year project 2").exists() else Path(".")
CSV_PATH = BASE_DIR / "datasets" / "lstm_all_scenarios.csv"
EMB_DIR = BASE_DIR / "artifacts" / "gat" / "spatial" / "embeddings"
EMB_INDEX_PATH = BASE_DIR / "artifacts" / "gat" / "spatial" / "embedding_index.csv"
OUTPUT_DIR = BASE_DIR / "artifacts" / "lstm_gat_fusion"
EMB_OUTPUT_DIR = OUTPUT_DIR / "embeddings"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
EMB_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "plots").mkdir(parents=True, exist_ok=True)

# Hyperparameters
WINDOW_SIZE = 30
STRIDE = 2
BATCH_SIZE = 64
EPOCHS = 20
LEARNING_RATE = 0.001
WEIGHT_DECAY = 0.0001
EARLY_STOPPING_PATIENCE = 5
SEED = 42

print("Configuration:")
print(f"  Input CSV:          {CSV_PATH}")
print(f"  Input GAT Emb Dir:  {EMB_DIR}")
print(f"  Output Dir:         {OUTPUT_DIR}")
print(f"  Output Emb Dir:     {EMB_OUTPUT_DIR}")
print(f"  Window Size: {WINDOW_SIZE}, Stride: {STRIDE}")

In [ ]:
# 1. Load Raw CSV Dataset
print(f"Loading raw dataset from {CSV_PATH}...")
if CSV_PATH.suffix == ".parquet":
    df_raw = pd.read_parquet(CSV_PATH)
else:
    df_raw = pd.read_csv(CSV_PATH)

print(f"Loaded {len(df_raw):,} raw rows across {len(df_raw['scenario'].unique())} scenarios.")

# 2. Load GAT Spatial Embedding Index
print(f"Loading GAT embedding metadata index from {EMB_INDEX_PATH}...")
df_emb_idx = pd.read_csv(EMB_INDEX_PATH)
print(f"Loaded {len(df_emb_idx):,} GAT embedding records.")

# Index GAT spatial embedding filepath by (scenario, timestep)
gat_emb_file_map = {}
for _, row in df_emb_idx.iterrows():
    gat_emb_file_map[(row["scenario"], row["timestep"])] = EMB_DIR / row["embedding_file"]

print(f"Mapped {len(gat_emb_file_map):,} GAT spatial snapshot files.")

# Select 24 non-duplicate raw numerical features
exclude_cols = {
    "scenario", "seed", "satellite_id", "timestep", "window_id", "step_in_window",
    "Unnamed: 0", "index", "congestion_score", "pos_ecef_z", "neighbor_count",
    "failure_indicator", "node_degree"
}
raw_feature_cols = [c for c in df_raw.columns if c not in exclude_cols and pd.api.types.is_numeric_dtype(df_raw[c])]
print(f"Selected {len(raw_feature_cols)} raw features: {raw_feature_cols[:5]}...")

In [ ]:
# Extract sequences by fusing 24 Raw Features + 128 GAT Spatial Embeddings -> 152-dim
train_samples = []
val_samples = []
test_samples = []
all_samples = []

print("Building time-aware sliding window sequences per (scenario, satellite_id)...")

scenarios = sorted(df_raw["scenario"].unique())
grouped = df_raw.groupby(["scenario", "satellite_id"])

for (scen, sat_id), g_df in grouped:
    g_sorted = g_df.sort_values("timestep").reset_index(drop=True)
    n_t = len(g_sorted)
    
    n_train_t = int(n_t * 0.70)  # 503
    n_val_t = int(n_t * 0.15)    # 108
    
    train_df = g_sorted.iloc[:n_train_t].reset_index(drop=True)
    val_df = g_sorted.iloc[n_train_t : n_train_t + n_val_t].reset_index(drop=True)
    test_df = g_sorted.iloc[n_train_t + n_val_t :].reset_index(drop=True)

    def extract_sequences_from_df(sub_df, split_name):
        seqs = []
        m = len(sub_df)
        if m <= WINDOW_SIZE:
            return seqs
            
        raw_mat = sub_df[raw_feature_cols].values.astype(np.float32)
        y_mat = sub_df["congestion_score"].values.astype(np.float32)
        t_mat = sub_df["timestep"].values.astype(int)

        for idx in range(0, m - WINDOW_SIZE, STRIDE):
            x_raw_win = raw_mat[idx : idx + WINDOW_SIZE]  # [30, 24]
            t_win = t_mat[idx : idx + WINDOW_SIZE]
            y_next = float(y_mat[idx + WINDOW_SIZE])
            y_curr = float(y_mat[idx + WINDOW_SIZE - 1])
            target_t = int(t_mat[idx + WINDOW_SIZE])

            # Load GAT spatial embeddings for this satellite across the 30-timestep window
            gat_embs = []
            for t in t_win:
                emb_file = gat_emb_file_map[(scen, t)]
                payload = torch.load(emb_file, weights_only=False)
                node_emb = payload["node_embeddings"][sat_id].numpy()  # [128]
                gat_embs.append(node_emb)

            gat_mat = np.vstack(gat_embs)  # [30, 128]
            x_fused = np.hstack([x_raw_win, gat_mat])  # [30, 24 + 128 = 152]

            item = {
                "x": x_fused,
                "y": y_next,
                "y_curr": y_curr,
                "scenario": scen,
                "satellite_id": sat_id,
                "target_t": target_t,
                "split": split_name
            }
            seqs.append(item)
        return seqs

    tr_s = extract_sequences_from_df(train_df, "train")
    va_s = extract_sequences_from_df(val_df, "val")
    te_s = extract_sequences_from_df(test_df, "test")

    train_samples.extend(tr_s)
    val_samples.extend(va_s)
    test_samples.extend(te_s)
    all_samples.extend(tr_s + va_s + te_s)

print(f"Sequence Generation Summary:")
print(f"  Train Sequences: {len(train_samples):,}")
print(f"  Val Sequences:   {len(val_samples):,}")
print(f"  Test Sequences:  {len(test_samples):,}")
print(f"  Total Sequences: {len(all_samples):,}")
print(f"  Input Sequence Shape: {train_samples[0]['x'].shape} (24 raw + 128 GAT = 152 features)")

In [ ]:
class FeatureScaler:
    def __init__(self):
        self.scaler = StandardScaler()
        self.fitted = False

    def fit(self, samples):
        all_x = np.vstack([s["x"] for s in samples])  # [N * W, F]
        self.scaler.fit(all_x)
        self.fitted = True

    def transform(self, x):
        if not self.fitted:
            raise RuntimeError("Scaler must be fitted!")
        if x.ndim == 2:
            return self.scaler.transform(x).astype(np.float32)
        elif x.ndim == 3:
            B, W, F = x.shape
            x_flat = x.reshape(-1, F)
            scaled = self.scaler.transform(x_flat)
            return scaled.reshape(B, W, F).astype(np.float32)

class TargetScaler:
    def __init__(self):
        self.scaler = StandardScaler()
        self.fitted = False

    def fit(self, samples):
        all_y = np.array([s["y"] for s in samples]).reshape(-1, 1)
        self.scaler.fit(all_y)
        self.fitted = True

    def transform(self, y):
        y_arr = np.array(y, dtype=np.float32).reshape(-1, 1)
        return self.scaler.transform(y_arr).astype(np.float32)

    def inverse_transform(self, y_scaled):
        y_np = y_scaled.cpu().numpy() if isinstance(y_scaled, torch.Tensor) else y_scaled
        if y_np.ndim == 1:
            y_np = y_np.reshape(-1, 1)
        return self.scaler.inverse_transform(y_np)

# Fit scalers strictly on training set
print("Fitting feature and target scalers strictly on training set...")
feature_scaler = FeatureScaler()
feature_scaler.fit(train_samples)

target_scaler = TargetScaler()
target_scaler.fit(train_samples)

with open(OUTPUT_DIR / "feature_scaler.pkl", "wb") as f:
    pickle.dump(feature_scaler, f)
with open(OUTPUT_DIR / "target_scaler.pkl", "wb") as f:
    pickle.dump(target_scaler, f)

print("[OK] Scalers saved to:", OUTPUT_DIR)

In [ ]:
class SequenceDataset(Dataset):
    def __init__(self, samples, feat_scaler, targ_scaler):
        self.samples = samples
        self.feat_scaler = feat_scaler
        self.targ_scaler = targ_scaler

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        x_scaled = self.feat_scaler.transform(s["x"])
        y_scaled = self.targ_scaler.transform(s["y"]).squeeze()
        
        x_tensor = torch.tensor(x_scaled, dtype=torch.float32)
        y_tensor = torch.tensor(y_scaled, dtype=torch.float32).unsqueeze(-1)
        y_raw_tensor = torch.tensor(s["y"], dtype=torch.float32).unsqueeze(-1)
        y_curr = float(s["y_curr"])

        return x_tensor, y_tensor, y_raw_tensor, y_curr, idx

train_ds = SequenceDataset(train_samples, feature_scaler, target_scaler)
val_ds = SequenceDataset(val_samples, feature_scaler, target_scaler)
test_ds = SequenceDataset(test_samples, feature_scaler, target_scaler)
all_ds = SequenceDataset(all_samples, feature_scaler, target_scaler)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)
all_loader = DataLoader(all_ds, batch_size=BATCH_SIZE, shuffle=False)

print(f"DataLoaders Prepared: Train: {len(train_loader)}, Val: {len(val_loader)}, Test: {len(test_loader)}")

In [ ]:
class LEOLSTMModel(nn.Module):
    def __init__(self, input_dim, hidden_dim=128, num_layers=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.head = nn.Sequential(
            nn.Linear(hidden_dim, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1),
        )

    def forward(self, x):
        lstm_out, (h_n, c_n) = self.lstm(x)
        h_last = lstm_out[:, -1, :]
        pred_congestion = self.head(h_last)
        return pred_congestion, h_last

input_dim = train_samples[0]["x"].shape[1]
model = LEOLSTMModel(input_dim=input_dim, hidden_dim=128, num_layers=2, dropout=0.2).to(device)

print(f"Initialized LEOLSTMModel (Input Dim: {input_dim}, Hidden Dim: 128, Num Layers: 2)")
print(model)

In [ ]:
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=3)

best_val_loss = float("inf")
history = []

print(f"Starting LSTM Training on {device} for {EPOCHS} epochs...")
print("-" * 75)

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    
    # Train
    model.train()
    train_loss = 0.0
    for x_b, y_b, _, _, _ in train_loader:
        x_b, y_b = x_b.to(device), y_b.to(device)
        optimizer.zero_grad()
        pred_b, _ = model(x_b)
        loss = criterion(pred_b, y_b)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * len(x_b)
    train_loss /= len(train_ds)

    # Validation
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for x_b, y_b, _, _, _ in val_loader:
            x_b, y_b = x_b.to(device), y_b.to(device)
            pred_b, _ = model(x_b)
            loss = criterion(pred_b, y_b)
            val_loss += loss.item() * len(x_b)
    val_loss /= len(val_ds)

    scheduler.step(val_loss)
    elapsed = time.time() - t0
    lr_curr = optimizer.param_groups[0]["lr"]

    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "lr": lr_curr,
        "duration_s": elapsed
    })

    print(f"Epoch {epoch:2d}/{EPOCHS:2d} | Train Loss (MSE): {train_loss:.6f} | Val Loss (MSE): {val_loss:.6f} | LR: {lr_curr:.6f} | Time: {elapsed:.2f}s")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save({"model_state_dict": model.state_dict()}, OUTPUT_DIR / "lstm_best.pt")

print("\n[OK] Model training completed. Best checkpoint saved to:", OUTPUT_DIR / "lstm_best.pt")

In [ ]:
# Load best model for evaluation
ckpt = torch.load(OUTPUT_DIR / "lstm_best.pt", weights_only=False)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()

y_true_raw = []
y_pred_raw = []
y_curr_raw = []

with torch.no_grad():
    for x_b, y_b_scaled, y_b_raw, y_curr, _ in test_loader:
        x_b = x_b.to(device)
        pred_scaled, _ = model(x_b)
        pred_raw = target_scaler.inverse_transform(pred_scaled)

        y_true_raw.extend(y_b_raw.numpy().flatten())
        y_pred_raw.extend(pred_raw.flatten())
        y_curr_raw.extend(y_curr.numpy().flatten() if isinstance(y_curr, torch.Tensor) else y_curr)

y_true_raw = np.array(y_true_raw)
y_pred_raw = np.array(y_pred_raw)
y_curr_raw = np.array(y_curr_raw)

# Calculate metrics on raw scale [0.0, 2.0]
test_mse = float(np.mean((y_true_raw - y_pred_raw) ** 2))
test_rmse = float(np.sqrt(test_mse))
test_mae = float(np.mean(np.abs(y_true_raw - y_pred_raw)))
var_true = float(np.var(y_true_raw))
test_r2 = float(1.0 - (test_mse / max(1e-8, var_true)))

# Persistence Baseline metrics (predict y(t+1) = y(t))
p_mse = float(np.mean((y_true_raw - y_curr_raw) ** 2))
p_rmse = float(np.sqrt(p_mse))
p_mae = float(np.mean(np.abs(y_true_raw - y_curr_raw)))
p_r2 = float(1.0 - (p_mse / max(1e-8, var_true)))

print("=" * 60)
print("FINAL TEST EVALUATION METRICS (RAW SCALE)")
print("=" * 60)
print(f"LSTM Test MSE:             {test_mse:.6f}")
print(f"LSTM Test RMSE:            {test_rmse:.6f} (RMSE^2 = {test_rmse**2:.6f})")
print(f"LSTM Test MAE:             {test_mae:.6f}")
print(f"LSTM Test R^2 Score:       {test_r2:.6f}")
print("-" * 60)
print(f"Persistence Baseline MAE:  {p_mae:.6f}")
print(f"Persistence Baseline R^2:  {p_r2:.6f}")
print(f"MAE Improvement over Persistence: +{((p_mae - test_mae) / p_mae) * 100:.2f}%")
print("=" * 60)

In [ ]:
# Export 128-dimensional Temporal Node Embeddings for PPO Reinforcement Learning
print("Exporting 128-dimensional temporal node embeddings for all sequences...")
model.eval()

exported_records = []

with torch.no_grad():
    for x_b, y_b_scaled, y_b_raw, _, idx_b in all_loader:
        x_b = x_b.to(device)
        pred_scaled, temporal_emb_b = model(x_b)  # temporal_emb_b shape: [batch_size, 128]
        pred_raw_b = target_scaler.inverse_transform(pred_scaled)
        
        temporal_emb_np = temporal_emb_b.cpu().numpy()  # [batch_size, 128]
        pred_raw_np = pred_raw_b.flatten()
        
        for b_i, global_idx in enumerate(idx_b.numpy()):
            sample_meta = all_samples[global_idx]
            scen = sample_meta["scenario"]
            sat_id = sample_meta["satellite_id"]
            target_t = sample_meta["target_t"]
            emb_vec = temporal_emb_np[b_i]  # 128-dim vector
            pred_y = float(pred_raw_np[b_i])
            true_y = float(sample_meta["y"])
            
            # Payload for downstream PPO RL agent
            payload = {
                "scenario": scen,
                "satellite_id": sat_id,
                "target_timestep": target_t,
                "temporal_embedding": emb_vec,  # [128]
                "predicted_congestion": pred_y,
                "true_congestion": true_y,
                "split": sample_meta["split"]
            }
            
            emb_filename = f"temporal_emb_{scen}_sat{sat_id:03d}_t{target_t:04d}.pt"
            emb_filepath = EMB_OUTPUT_DIR / emb_filename
            torch.save(payload, emb_filepath)
            
            exported_records.append({
                "scenario": scen,
                "satellite_id": sat_id,
                "target_timestep": target_t,
                "embedding_file": emb_filename,
                "predicted_congestion": pred_y,
                "true_congestion": true_y,
                "split": sample_meta["split"]
            })

df_temporal_idx = pd.DataFrame(exported_records)
df_temporal_idx.to_csv(OUTPUT_DIR / "temporal_embedding_index.csv", index=False)

print(f"[OK] Exported {len(exported_records):,} temporal 128-D node embedding files to:", EMB_OUTPUT_DIR)
print(f"Master Metadata Index saved to:", OUTPUT_DIR / "temporal_embedding_index.csv")
print(f"Sample Temporal Embedding Shape: {exported_records[0]['temporal_embedding'].shape} (128-D vector)")

In [ ]:
# Visualization Plots
# 1. Training & Validation Loss Curve
plt.figure(figsize=(8, 5), dpi=300)
epochs_arr = [h["epoch"] for h in history]
plt.plot(epochs_arr, [h["train_loss"] for h in history], label="Train MSE Loss", color="#1f77b4", linewidth=2)
plt.plot(epochs_arr, [h["val_loss"] for h in history], label="Val MSE Loss", color="#ff7f0e", linewidth=2, linestyle="--")
plt.title("LSTM Training & Validation Loss Convergence", fontsize=12, fontweight="bold")
plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "plots" / "lstm_loss_curve.png")
plt.show()

# 2. Actual vs Predicted Congestion Scatter Plot
plt.figure(figsize=(7, 7), dpi=300)
plt.scatter(y_true_raw[::10], y_pred_raw[::10], alpha=0.3, s=10, color="#2ca02c")
plt.plot([0, 2], [0, 2], 'k--', label="Ideal 1:1 Line")
plt.title(f"LSTM Actual vs Predicted Congestion (R^2 = {test_r2:.4f})", fontsize=12, fontweight="bold")
plt.xlabel("Ground-Truth Congestion Score (t+1)")
plt.ylabel("Predicted Congestion Score (t+1)")
plt.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "plots" / "actual_vs_predicted.png")
plt.show()

# 3. 2D PCA Scatter Plot of Output 128-D Temporal Embeddings
sample_embs = np.vstack([r["temporal_embedding"] for r in exported_records[:2000]])
pca = PCA(n_components=2)
embs_2d = pca.fit_transform(sample_embs)

plt.figure(figsize=(8, 6), dpi=300)
plt.scatter(embs_2d[:, 0], embs_2d[:, 1], c=[r["predicted_congestion"] for r in exported_records[:2000]], cmap="viridis", alpha=0.6, s=15)
plt.colorbar(label="Predicted Congestion Score (t+1)")
plt.title(f"2D PCA Projection of Output 128-D LSTM Temporal Embeddings", fontsize=12, fontweight="bold")
plt.xlabel("PCA Component 1")
plt.ylabel("PCA Component 2")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "plots" / "temporal_embedding_pca.png")
plt.show()